# Lab 13: Pose Estimation, an Exercise Counter

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
So far the model drew a **box** around a person. A **pose estimation** model does more: it finds the **body points** of the person (nose, shoulders, elbows, wrists, hips, knees and ankles) and joins them into a **skeleton**. With the points you can measure things: Is the hand above the shoulder? How much is the knee bent? In this lab you use the angle of a joint to **count exercise repetitions** (squats or arm curls) and to detect a raised hand.

**By the end of this lab you can:**
1. Explain what a keypoint is and name the 17 points of the COCO body model.
2. Read the x, y position and the confidence of each keypoint from the model result.
3. Calculate the angle of a joint from three keypoints.
4. Count repetitions with a simple two-state rule (open and closed).
5. Test the counter and explain why it sometimes miscounts.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Privacy.** The camera sees faces and bodies. Use it only on yourself and on classmates who agree. The program does not save pictures.

## 2. Background

**Keypoints.** The model `yolo26n-pose` finds up to 17 body points for each person. Each point has an **x** and a **y** position in the picture (in pixels, with (0, 0) at the top left, so y grows **downwards**) and a **confidence** from 0 to 1 that tells how sure the model is that the point is visible.

| No. | Point | No. | Point | No. | Point |
|---|---|---|---|---|---|
| 0 | nose | 6 | right shoulder | 12 | right hip |
| 1 | left eye | 7 | left elbow | 13 | left knee |
| 2 | right eye | 8 | right elbow | 14 | right knee |
| 3 | left ear | 9 | left wrist | 15 | left ankle |
| 4 | right ear | 10 | right wrist | 16 | right ankle |
| 5 | left shoulder | 11 | left hip | | |

"Left" and "right" are the left and right of the **person**, not of the picture.

**Pose model and detector model.** The pose model also draws a box around each person, so it does the job of the detector of Lab 10 and finds the points in one step. It is a bit slower than the detector, about 25 frames per second on your Jetson. It is fast enough for this lab without TensorRT (building the pose engine takes about 10 minutes, so this lab does not use it).

**The angle of a joint.** Take three points: A, B and C. B is the joint (for the elbow: A = shoulder, B = elbow, C = wrist). The angle at B between the lines BA and BC is the **joint angle**.

| Joint | A | B | C | Straight | Bent |
|---|---|---|---|---|---|
| Elbow (arm curl) | shoulder | elbow | wrist | about 170 degrees | about 40 degrees |
| Knee (squat) | hip | knee | ankle | about 170 degrees | about 90 degrees |

The program calculates the angle with `math.atan2`, which gives the direction of a line from its x and y difference. The angle at B is the difference between the directions of BA and BC.

**Counting a repetition.** A repetition has two states: **open** (the joint is straight) and **closed** (the joint is bent). A rule with two limits counts one repetition each time the joint goes from open to closed:

| Angle | Rule |
|---|---|
| above `OPEN_ANGLE` | the state becomes **open** |
| below `CLOSE_ANGLE` **and** the state was open | the state becomes **closed** and the count goes up by 1 |
| between the two limits | nothing changes |

There is a gap between the two limits on purpose. Like the margin of Lab 14, it stops the counter from counting many times when the angle shakes near one limit.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(13)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (15 min)

### Step 1: Get the Pose Model
The model file `yolo26n-pose.pt` is small (8 MB). The cell uses the copy in the `models` folder, or downloads it the first time (this needs the internet).

In [ ]:
import glob
import os
import shutil

print("This notebook is saved in:", os.getcwd())

for source in ["../models/yolo26n-pose.pt", "../lab14/yolo26n-pose.pt"]:
    if os.path.exists(source) and not os.path.exists("yolo26n-pose.pt"):
        shutil.copy(source, "yolo26n-pose.pt")
        print("copied", source)

if os.path.exists("yolo26n-pose.pt"):
    print("The pose model is here.")
else:
    print("The model will be downloaded when you load it in Part B (needs the internet).")

### Step 2: Plan Your Exercise
Choose an exercise and think about where the camera must stand. The camera must see **the whole joint** from the **side**: a side view shows the bend of the knee or elbow, a front view hides it.

| Exercise | Joint | Camera sees | Start | Count when |
|---|---|---|---|---|
| Arm curl | elbow | your side, from the shoulder to the wrist | arm straight | arm bent (angle below `CLOSE_ANGLE`) |
| Squat | knee | your side, from the hip to the ankle | standing | knees bent (angle below `CLOSE_ANGLE`) |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Exercise you chose:
- Where will the camera be and how far from you?:
- Why is a side view better than a front view for this joint?:


---
# Part B: See the Skeleton (35 min)

### Step 1: The Settings
Words in capital letters are settings you can change. Run the cell.

In [ ]:
# ---------------- Settings you can change ----------------
EXERCISE = "curl"        # "curl" (elbow) or "squat" (knee)
CLOSE_ANGLE = 60         # below this angle the joint is closed (a repetition is counted)
OPEN_ANGLE = 150         # above this angle the joint is open (ready for the next repetition)
CONFIDENCE = 0.40        # confidence limit of a person
POINT_CONFIDENCE = 0.50  # confidence limit of a keypoint (lower points are ignored)
MODEL = "yolo26n-pose.pt"
# ----------------------------------------------------------
print("Exercise:", EXERCISE, "| closed below", CLOSE_ANGLE, "| open above", OPEN_ANGLE)

### Step 2: Load the Model

In [ ]:
import csv
import math
import time
from datetime import datetime

import cv2
import torch
from ultralytics import YOLO

import labview

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL)
print("Using model:", MODEL, "on", "the GPU" if device == 0 else "the CPU")

### Step 3: Read the Keypoints from a Picture
Before the live video, look at the numbers. The cell runs the model on a picture of people and prints the 17 points of the first person: the position in pixels and the confidence. `result.keypoints.xy` holds the positions of all people (shape: people, 17, 2) and `result.keypoints.conf` holds the confidences (shape: people, 17).

In [ ]:
NAMES = ["nose", "left eye", "right eye", "left ear", "right ear", "left shoulder", "right shoulder",
         "left elbow", "right elbow", "left wrist", "right wrist", "left hip", "right hip",
         "left knee", "right knee", "left ankle", "right ankle"]

picture = next((p for p in ("../models/bus.jpg", "../lab15/bus.jpg") if os.path.exists(p)), None)
if picture is None:
    print("No sample picture found. Skip this step.")
else:
    result = model(picture, conf=CONFIDENCE, device=device, verbose=False)[0]
    xy, conf = result.keypoints.xy.cpu().numpy(), result.keypoints.conf.cpu().numpy()
    print("People found:", len(xy), "| points per person:", xy.shape[1])
    print()
    print(f"{'No.':<4}{'point':<16}{'x':>6}{'y':>6}{'confidence':>12}")
    for i, name in enumerate(NAMES):
        print(f"{i:<4}{name:<16}{xy[0][i][0]:>6.0f}{xy[0][i][1]:>6.0f}{conf[0][i]:>12.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many people did the model find in the picture?:
- What are x and y of the nose of the first person?:
- Which point has the lowest confidence? Why do you think so?:
- A point that is hidden has x and y near 0. Why must the program check the confidence first?:


### Step 4: The Skeleton Live
The function `result.plot(boxes=False)` draws the skeleton for you. Run the cell, stand so that the camera sees your whole upper body and move your arms. Watch how the points follow you. Try a raised hand, a turned body and a hand hidden behind your back. Press **Stop** when you are done.

In [ ]:
def process_skeleton(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = result.plot(boxes=False)
    cv2.putText(shown, f"People: {len(result.boxes)}   {state['fps']:.0f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    return shown


live = labview.live(process_skeleton, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- What happens to the points of an arm that you hide behind your back?:
- What happens when two people stand close together?:
- Which points are the most stable and which jump the most?:


---
# Part C: Measure Angles and Count Repetitions (55 min)

### Step 1: The Program
Read this cell before you run it. The new parts are:

| Function | What it does |
|---|---|
| `angle(a, b, c)` | Gives the angle at point b in degrees (0 to 180) from three (x, y) points |
| `joint_angle(xy, conf)` | Picks the left or right side (the one the camera sees better) and returns the angle of the joint of your `EXERCISE`, or `None` if a point is not clear |
| `update_reps(deg)` | The two-state rule: counts one repetition when the state goes from open to closed |
| `hand_up(xy, conf)` | `True` when a wrist is **above** its shoulder (a smaller y value, because y grows downwards) |
| `log_rep(rep, deg)` | Adds a line to `reps.csv` and rewrites `reps.xlsx` |
| `process(frame, state)` | Runs the model, takes the biggest person, measures, counts and draws. The live view calls it for every picture. |

Find the five steps of a vision application again: capture (live view), detect (the model finds the points), decide (`update_reps`), act (the drawing) and record (`log_rep`).

In [ ]:
# keypoint numbers (a, b, c) of each joint, left side then right side
JOINTS = {"curl": ((5, 7, 9), (6, 8, 10)),        # shoulder, elbow, wrist
          "squat": ((11, 13, 15), (12, 14, 16))}  # hip, knee, ankle

run = {"state": None, "reps": 0, "last_rep": None, "note": "", "warned": False}


def angle(a, b, c):
    """The angle at point b between the lines b-a and b-c, in degrees (0 to 180)."""
    first = math.atan2(a[1] - b[1], a[0] - b[0])
    second = math.atan2(c[1] - b[1], c[0] - b[0])
    deg = abs(math.degrees(first - second))
    return 360 - deg if deg > 180 else deg


def joint_angle(xy, conf):
    """Angle of the joint of EXERCISE. Uses the side whose three points are the clearest. None if no side is clear."""
    best = None
    for side in JOINTS[EXERCISE]:
        worst = min(conf[i] for i in side)               # the weakest point of this side
        if worst >= POINT_CONFIDENCE and (best is None or worst > best[0]):
            best = (worst, side)
    if best is None:
        return None
    a, b, c = (xy[i] for i in best[1])
    return angle(a, b, c)


def update_reps(deg):
    """The two-state rule. Returns True when this angle completes a repetition."""
    if deg is None:
        return False
    if deg > OPEN_ANGLE:
        run["state"] = "open"
    elif deg < CLOSE_ANGLE and run["state"] == "open":
        run["state"] = "closed"
        run["reps"] += 1
        return True
    return False


def hand_up(xy, conf):
    """True when a wrist is above its shoulder (y is smaller). Both points must be clear."""
    for shoulder, wrist in ((5, 9), (6, 10)):
        if min(conf[shoulder], conf[wrist]) >= POINT_CONFIDENCE and xy[wrist][1] < xy[shoulder][1]:
            return True
    return False


def write_excel():
    """Rewrite reps.xlsx from reps.csv, so it opens in Excel with a header and number columns."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("reps.xlsx needs openpyxl. reps.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Repetitions"
    with open("reps.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[2], row[3], row[4] = int(row[2]), float(row[3]), int(row[4])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCDE", (20, 10, 8, 14, 18)):
        sheet.column_dimensions[column].width = width
    book.save("reps.xlsx")


def log_rep(rep, deg):
    """Add one line to the log files (reps.csv and reps.xlsx)."""
    now = time.time()
    seconds = 0.0 if run["last_rep"] is None else now - run["last_rep"]
    run["last_rep"] = now
    first = not os.path.exists("reps.csv")
    with open("reps.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "exercise", "rep", "seconds_since_last", "angle_at_count"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), EXERCISE, rep, round(seconds, 1), round(deg)])
    write_excel()


def process(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = result.plot(boxes=False)
    deg = None
    raised = False
    if len(result.boxes) > 0:
        boxes = result.boxes.xyxy
        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        person = int(areas.argmax())                       # the biggest person is the one who exercises
        xy = result.keypoints.xy[person].cpu().numpy()
        conf = result.keypoints.conf[person].cpu().numpy()
        deg = joint_angle(xy, conf)
        raised = hand_up(xy, conf)
        if update_reps(deg):
            log_rep(run["reps"], deg)
            run["note"] = f"Rep {run['reps']} at {datetime.now().strftime('%H:%M:%S')}"

    cv2.putText(shown, f"{EXERCISE}  reps: {run['reps']}", (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
    angle_text = "angle: ---" if deg is None else f"angle: {deg:.0f}  ({run['state'] or 'start'})"
    cv2.putText(shown, angle_text, (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    if raised:
        cv2.putText(shown, "HAND UP", (10, 105), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The program is ready.")

### Step 2: Check the Angle Maths
Before the live test, check `angle` with points you know. Three points that form a right angle must give 90 degrees, and three points on one line must give 180 degrees.

In [ ]:
print("Right angle:   ", round(angle((0, 0), (0, 10), (10, 10))), "(expected 90)")
print("Straight line: ", round(angle((0, 0), (5, 5), (10, 10))), "(expected 180)")
print("Sharp bend:    ", round(angle((10, 0), (0, 0), (10, 2))), "(expected about 11)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did all three lines give the expected numbers?:
- Why does `angle` return 360 - deg when deg is above 180?:


### Step 3: Run the Counter
Run the cell and start your exercise. Watch the angle and the state under the repetition count. Do five slow repetitions. A repetition counts when the angle goes **above `OPEN_ANGLE`** (open) and then **below `CLOSE_ANGLE`** (closed). A hand above the shoulder shows `HAND UP`. Press **Stop** when you are done.

If the camera does not see the whole joint, the angle shows `---` and nothing is counted. Move or turn so that the shoulder, elbow and wrist (or hip, knee and ankle) are all visible.

In [ ]:
run.update(state=None, reps=0, last_rep=None, note="")
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

### Step 4: See the Results
Run this cell after you pressed Stop. It shows the log and the average time of one repetition. To open the Excel file, use the file browser on the left of JupyterLab, right-click `reps.xlsx` and choose **Download**, then open it on your laptop.

In [ ]:
def show_results():
    for name in ("reps.csv", "reps.xlsx"):
        print(f"{name:<12}", "found" if os.path.exists(name) else "not created yet (it appears at the first repetition)")
    if os.path.exists("reps.csv"):
        with open("reps.csv", newline="") as f:
            rows = list(csv.DictReader(f))
        times = [float(r["seconds_since_last"]) for r in rows[1:]]
        print()
        print("Repetitions in the log:", len(rows))
        if times:
            print(f"Average time of one repetition: {sum(times) / len(times):.1f} seconds")
        print()
        with open("reps.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of repetitions you did and the number the program counted:
- The angle shown when your joint was fully open:
- The angle shown when your joint was fully closed:
- Do your angles match the table in the Background section? If not, what are your own limits?:


---
# Part D: Make It Your Own (35 min)
First set `OPEN_ANGLE` and `CLOSE_ANGLE` to match **your own** open and closed angles from Part C (leave a gap of at least 40 degrees). Then add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run Step 3 of Part C again.

- Change `EXERCISE` to `"squat"` and test it from the side.
- Add a new exercise to `JOINTS`, for example the **shoulder** (hip, shoulder, elbow) for arm raises to the side.
- Show a target on the screen, for example `reps: 3 of 10`, and write `GOAL!` when you reach it.
- Count the **hand raises** (use `hand_up`) with the same two-state idea: a raise is counted when the hand goes up after it was down.
- Warn the person with the text `GO LOWER` when the joint opens again before the angle went below `CLOSE_ANGLE` (a half repetition).
- Show the repetitions per minute on the screen.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your own open and closed angles:
- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part E: Test and Present (30 min)

### Step 1: Measure the Accuracy
Delete the old log, then do **10 real repetitions** in each situation and write what the program counted. One person exercises, the other counts aloud and watches the screen.

In [ ]:
# Delete the old log to start a clean test
for name in ("reps.csv", "reps.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("reps*")))

Run Step 3 of Part C for each test (it starts new counts).

| Test | Situation | Real | Program |
|---|---|---|---|
| 1 | Side view, slow and full repetitions | 10 | |
| 2 | Side view, fast repetitions | 10 | |
| 3 | Side view, half repetitions (the joint does not bend enough) | 10 | |
| 4 | Front view (the camera faces you) | 10 | |
| 5 | A second person walks behind you | 10 | |

**Accuracy** = `100 - (|real - program| / real × 100)`. For example, 10 real and 9 counted gives `100 - 10 = 90 %`.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 result and accuracy:
- Test 2 result and accuracy:
- Test 3: was a half repetition counted? Is this right?:
- Test 4: what happened with the front view and why?:
- Test 5: what happened with the second person?:
- OPEN_ANGLE and CLOSE_ANGLE that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration** to your instructor: show the skeleton, do the exercise, show the count, the angle and the log in Excel, and explain how a repetition is counted.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Exercise and camera position:
- Model file used and the FPS shown:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- Overall accuracy of your counter:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** A pose counter helps with fitness coaching, physiotherapy and safety checks (a person who falls, or a worker who bends the wrong way). The next lab uses a model that finds the **exact shape** of objects, not only a box.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The skeleton shows live and follows your movement | |
| 2 | The keypoint table from Part B Step 3 is explained (x, y, confidence) | |
| 3 | The three angle checks of Part C Step 2 give the expected numbers | |
| 4 | The live counter shows the angle, the state and the repetitions | |
| 5 | Five repetitions are counted correctly, and `HAND UP` shows when a hand is raised | |
| 6 | The log is shown in Excel (`reps.xlsx`) with at least 4 lines | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table in Part E is complete, with the accuracy calculated | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is a keypoint? How many does the COCO pose model find for each person?:
- 2. Why does the program check the confidence of a keypoint before it uses the point?:
- 3. In the picture, y grows downwards. How does this change the test for a raised hand?:
- 4. What three points do you use for the angle of the elbow, and which of them is the joint?:
- 5. Why does the counter use two limits (`OPEN_ANGLE` and `CLOSE_ANGLE`) and not one?:
- 6. A person does a half repetition and the counter does not count it. Is the counter wrong? Explain.:
- 7. Why is a side view better than a front view for counting squats?:
- 8. The counter counts 8 when you did 10. What is the accuracy? Name two possible causes.:
- 9. Which is a good use of pose estimation? (a) Reading a number plate (b) Checking that a worker lifts with a straight back (c) Counting cars (d) Finding a lost phone:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The model cannot be downloaded | No internet on the Jetson | Ask your teacher. Copy `yolo26n-pose.pt` to the `models` folder. |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| No skeleton appears | The person is too small or too dark | Stand closer, add light, lower the confidence slider |
| The angle shows `---` | A point of the joint is hidden or unclear | Turn so that the whole joint is visible. Lower `POINT_CONFIDENCE` a little. |
| The counter never counts | The angle never goes above `OPEN_ANGLE` or below `CLOSE_ANGLE` | Read your angles on the screen and set the limits to match them |
| The counter counts too much | The gap between the limits is too small, or the points shake | Make the gap bigger. Wear clothes that differ from the wall. |
| The wrong person is counted | The biggest person in the picture is not you | Stand closest to the camera, or ask others to move away |
| Left and right look swapped | The words are for the person, not for the picture | This is normal. The program uses the side that is clearer. |
| `CUDA out of memory` | Other notebooks and programs use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(13)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Pose estimation task: https://docs.ultralytics.com/tasks/pose/
- Ultralytics, Workouts monitoring: https://docs.ultralytics.com/guides/workouts-monitoring/
- COCO keypoints format: https://cocodataset.org/#keypoints-2020
- Python documentation, math.atan2: https://docs.python.org/3/library/math.html#math.atan2